# Subword Tokenization: BPE, WordPiece, SentencePiece+BPE, SentencePiece+Unigram

We will use the same small corpus for all four approaches:

    low lower lowest
    low lower

The important distinction is:

- **BPE** → repeatedly merges frequent pairs.
- **WordPiece** → learns useful vocabulary pieces using a likelihood/objective-based criterion.
- **SentencePiece + BPE** → SentencePiece framework + BPE merging.
- **SentencePiece + Unigram** → SentencePiece framework + probabilistic vocabulary/segmentation.

---

# 1. BPE

## Training

BPE starts with small units, usually characters.

For example:

    low    → l o w
    lower  → l o w e r
    lowest → l o w e s t

It counts adjacent pairs and finds the most frequent pair.

Suppose:

    l + o

is the most frequent pair.

It merges them:

    l + o → lo

Now:

    low    → lo w
    lower  → lo w e r
    lowest → lo w e s t

BPE counts pairs again.

Suppose:

    lo + w

is now the most frequent useful pair.

It merges:

    lo + w → low

Now:

    low    → low
    lower  → low e r
    lowest → low e s t

It continues this process:

    find frequent pair
        ↓
    merge pair
        ↓
    find next frequent pair
        ↓
    merge again
        ↓
    repeat

Eventually, BPE learns a vocabulary and a set of merge rules/ranks.

For example, it might learn pieces such as:

    low
    er
    est
    ...

## Processing new text

After training, suppose we give BPE:

    lower

It does NOT train again.

It starts with:

    l o w e r

and applies the learned merge rules:

    l + o → lo
    lo + w → low
    e + r → er

Final tokenization:

    lower → low + er

### Key idea

BPE:

    small pieces
        ↓
    repeatedly merge frequent pairs
        ↓
    learned vocabulary + merge rules
        ↓
    use those learned rules on new text


# 2. WordPiece

## Training

WordPiece also starts with small pieces.

For example:

    low
    lower
    lowest

can initially be represented using small pieces such as:

    l
    o
    w
    ##e
    ##r
    ##s
    ##t

The `##` means that the piece is a continuation of the same word.

WordPiece does NOT simply choose the most frequent pair like BPE.

Instead, it evaluates candidate pieces using its training objective/likelihood-based criterion and learns pieces that are useful for representing the corpus.

For example, it may learn:

    low
    ##er
    ##est
    ...

Eventually its vocabulary might contain:

    low
    ##er
    ##est
    l
    o
    w
    ##e
    ##r
    ...

## Processing new text

Suppose we give WordPiece:

    lower

It looks in its already-learned vocabulary.

It finds:

    low

Then the remaining part is:

    er

Because this is inside the word, WordPiece looks for a continuation token:

    ##er

It finds it.

Therefore:

    lower → low + ##er

Another example:

    lowest → low + ##est

WordPiece does NOT create `##er` or `##est` while processing the new sentence.

Those pieces were learned and stored in the vocabulary during tokenizer training.

### Key idea

WordPiece:

    corpus
        ↓
    evaluate useful vocabulary pieces
        ↓
    learn vocabulary
        ↓
    new word
        ↓
    find pieces in learned vocabulary
        ↓
    tokenize


# 3. SentencePiece + BPE

SentencePiece is a tokenizer framework/toolkit.

Here we are using the BPE algorithm inside SentencePiece.

## Training

SentencePiece can work directly with raw text.

Our corpus:

    low lower lowest
    low lower

SentencePiece represents word boundaries using `▁`.

So conceptually:

    ▁low ▁lower ▁lowest
    ▁low ▁lower

Then the BPE algorithm starts with small pieces.

For example:

    ▁ l o w
    ▁ l o w e r
    ▁ l o w e s t

BPE then repeatedly merges frequent pairs.

For example:

    l + o → lo

Then:

    lo + w → low

Then perhaps:

    e + r → er

The exact learned vocabulary depends on the corpus and training settings.

Eventually it may learn pieces such as:

    ▁low
    er
    est
    ...

## Processing new text

Suppose the new text is:

    low lower

SentencePiece first represents the boundaries:

    ▁low ▁lower

Then it applies the learned BPE merge rules.

For example:

    ▁low → ▁low

and:

    ▁lower → ▁low + er

Final tokenization could therefore look like:

    ▁low | ▁low | er

### Key idea

SentencePiece + BPE:

    raw text
        ↓
    SentencePiece handles text/boundaries
        ↓
    BPE repeatedly merges pieces
        ↓
    learned vocabulary + merge rules
        ↓
    apply them to new text


# 4. SentencePiece + Unigram

Here SentencePiece is used with the Unigram algorithm.

The biggest difference from BPE is:

    BPE → repeatedly MERGES

    Unigram → starts with MANY possible pieces
             and PRUNES less useful pieces

## Training

Our corpus:

    low lower lowest
    low lower

SentencePiece represents boundaries:

    ▁low ▁lower ▁lowest
    ▁low ▁lower

Unigram starts with many candidate pieces.

For example, candidates might include:

    ▁l
    ▁lo
    ▁low
    ▁lower
    ▁lowest
    low
    er
    est
    ...

There can be many candidate pieces in the initial vocabulary.

Unigram assigns probabilities/scores to these pieces.

It then determines which pieces are useful for explaining the corpus.

Less useful pieces are removed/pruned.

This process is repeated until the desired vocabulary size is reached.

Eventually the learned vocabulary might contain:

    ▁low
    ▁lower
    ▁lowest
    er
    est
    ...

## Processing new text

Suppose the new text is:

    lower

SentencePiece represents it as:

    ▁lower

Unigram can consider multiple possible segmentations.

For example:

    Option 1:
    ▁lower

    Option 2:
    ▁low + er

    Option 3:
    ▁l + o + w + e + r

The Unigram model calculates the probability/score of these possible segmentations.

Suppose:

    P(▁lower) = 0.02
    P(▁low + er) = 0.15
    P(▁l + o + w + e + r) = 0.001

Then:

    ▁low + er

has the highest score.

So the final tokenization is:

    lower → ▁low + er

### Key idea

SentencePiece + Unigram:

    raw text
        ↓
    SentencePiece handles text/boundaries
        ↓
    start with many candidate pieces
        ↓
    learn probabilities
        ↓
    prune less useful pieces
        ↓
    final vocabulary + probabilities
        ↓
    new text
        ↓
    consider possible segmentations
        ↓
    choose the most probable segmentation


# Final Comparison

| Method | Training idea | New text processing |
|---|---|---|
| BPE | Repeatedly merge frequent pairs | Apply learned merge rules |
| WordPiece | Learn useful pieces using its objective/likelihood criterion | Find pieces in learned vocabulary |
| SentencePiece + BPE | SentencePiece + repeated BPE merging | Apply learned BPE rules |
| SentencePiece + Unigram | Start with many pieces, learn probabilities, prune pieces | Choose the best/probable segmentation |

## Most important mental model

Every trained tokenizer has a learned vocabulary.

The algorithms are different ways of learning/using that vocabulary:

    BPE
    → merge frequent pairs

    WordPiece
    → learn useful pieces using its objective

    SentencePiece + BPE
    → SentencePiece framework + BPE merging

    SentencePiece + Unigram
    → SentencePiece framework + probabilistic segmentation

And the overall workflow is:

    TRAINING

    corpus
       ↓
    tokenizer training
       ↓
    learned vocabulary + model/rules
       ↓

    INFERENCE

    new sentence
       ↓
    use the already-trained tokenizer
       ↓
    subword tokens
       ↓
    token IDs


## One-line memory trick

    BPE       = MERGE
    WordPiece = LEARN USEFUL PIECES
    Unigram   = PROBABILITY + PRUNING
    SentencePiece = FRAMEWORK that can use BPE or Unigram